# 👷 Notebook 3: Worker Implementation

Build workers that pull jobs from the queue and process them.

## Learning Objectives

By the end of this notebook, you'll understand:
- Worker polling patterns
- Processing jobs safely
- Updating job status
- Running multiple workers

## 🔧 Setup

Before running anything, make sure:

1. Docker services are up: `docker compose up -d` from the lab root.
2. Dependencies are installed: `uv sync` from the lab root.
3. **Kernel is the lab `.venv`**: click the kernel picker in the top-right of this notebook and pick the interpreter under `.venv/bin/python`.
4. If you don't see it, reload the VS Code window: `Cmd+Shift+P` → *Reload Window*, then re-open the notebook.

In [ ]:
import redis
import psycopg2
import json
import uuid
import time
import random
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
from typing import Optional, Callable

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="taskqueue", user="postgres", password="postgres"
)
conn.autocommit = True

QUEUE_NAME = "jobs:pending"

print("✅ Connected to Redis and PostgreSQL!")

### 🌍 Real-world

YouTube runs thousands of transcoding workers in parallel — each worker pulls one video at a time and produces multiple resolutions.

## 🔄 Reset State

Run this cell to clear all jobs, logs, and queue data before running demos.

In [ ]:
def reset_all():
    r.flushall()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM job_logs")
    cursor.execute("DELETE FROM dead_letter_queue")
    cursor.execute("DELETE FROM jobs")
    cursor.close()
    print("🔄 Reset complete: Redis flushed, jobs/logs/dlq cleared")

reset_all()

## 👷 Worker Architecture

In [ ]:
print("👷 How Workers Work")
print("=" * 60)
print("""
Workers run in a continuous loop:

┌─────────────────────────────────────────────────────────────┐
│                     WORKER LOOP                             │
│                                                             │
│  while True:                                                │
│      │                                                      │
│      ▼                                                      │
│  ┌─────────────────────────────────────────┐               │
│  │ 1. BRPOP (blocking pop from queue)      │               │
│  │    Waits until job available            │               │
│  └────────────────────┬────────────────────┘               │
│                       │ got job_id                         │
│                       ▼                                    │
│  ┌─────────────────────────────────────────┐               │
│  │ 2. Fetch job details from database      │               │
│  │    UPDATE status = 'processing'         │               │
│  └────────────────────┬────────────────────┘               │
│                       │                                    │
│                       ▼                                    │
│  ┌─────────────────────────────────────────┐               │
│  │ 3. Execute the actual work              │               │
│  │    (generate PDF, transcode, etc.)      │               │
│  └────────────────────┬────────────────────┘               │
│                       │                                    │
│                       ▼                                    │
│  ┌─────────────────────────────────────────┐               │
│  │ 4. Update status                        │               │
│  │    'completed' or 'failed'              │               │
│  └────────────────────┬────────────────────┘               │
│                       │                                    │
│                       └──────────► loop                    │
└─────────────────────────────────────────────────────────────┘

BRPOP is key: It BLOCKS until a job is available.
No busy-waiting, no polling delays!
""")

## 🔧 Worker Implementation

In [ ]:
class Worker:
    def __init__(self, worker_id: str, redis_client, db_conn):
        self.worker_id = worker_id
        self.redis = redis_client
        self.conn = db_conn
        self.running = False
        self.jobs_processed = 0
        self.handlers = {}
    
    def register_handler(self, job_type: str, handler: Callable):
        self.handlers[job_type] = handler
    
    def fetch_job(self, job_id: str) -> Optional[dict]:
        cursor = self.conn.cursor()
        cursor.execute("""
            SELECT id, job_type, payload, attempts, max_attempts
            FROM jobs WHERE id = %s AND status = 'pending'
        """, (job_id,))
        row = cursor.fetchone()
        cursor.close()
        
        if not row:
            return None
        
        return {
            'id': str(row[0]),
            'job_type': row[1],
            'payload': row[2],
            'attempts': row[3],
            'max_attempts': row[4]
        }
    
    def update_status(self, job_id: str, status: str, result: dict = None, error: str = None):
        cursor = self.conn.cursor()
        
        if status == 'processing':
            cursor.execute("""
                UPDATE jobs 
                SET status = 'processing', worker_id = %s, 
                    started_at = NOW(), attempts = attempts + 1,
                    updated_at = NOW()
                WHERE id = %s
            """, (self.worker_id, job_id))
        elif status == 'completed':
            cursor.execute("""
                UPDATE jobs 
                SET status = 'completed', result = %s,
                    completed_at = NOW(), updated_at = NOW()
                WHERE id = %s
            """, (json.dumps(result), job_id))
        elif status == 'failed':
            cursor.execute("""
                UPDATE jobs 
                SET status = 'failed', error_message = %s,
                    completed_at = NOW(), updated_at = NOW()
                WHERE id = %s
            """, (error, job_id))
        
        cursor.close()
    
    def log_event(self, job_id: str, event: str, message: str):
        cursor = self.conn.cursor()
        cursor.execute("""
            INSERT INTO job_logs (job_id, event, message, worker_id)
            VALUES (%s, %s, %s, %s)
        """, (job_id, event, message, self.worker_id))
        cursor.close()
    
    def process_job(self, job: dict) -> bool:
        job_id = job['id']
        job_type = job['job_type']
        
        self.update_status(job_id, 'processing')
        self.log_event(job_id, 'started', f'Worker {self.worker_id} started processing')
        
        handler = self.handlers.get(job_type)
        if not handler:
            self.update_status(job_id, 'failed', error=f'No handler for job type: {job_type}')
            return False
        
        try:
            result = handler(job['payload'])
            self.update_status(job_id, 'completed', result=result)
            self.log_event(job_id, 'completed', 'Job completed successfully')
            return True
        except Exception as e:
            self.update_status(job_id, 'failed', error=str(e))
            self.log_event(job_id, 'failed', str(e))
            return False
    
    def run_once(self, timeout: int = 1) -> bool:
        result = self.redis.brpop(QUEUE_NAME, timeout=timeout)
        
        if not result:
            return False
        
        _, job_id = result
        
        job = self.fetch_job(job_id)
        if not job:
            return False
        
        self.process_job(job)
        self.jobs_processed += 1
        return True

print("✅ Worker class defined!")

## 🔨 Job Handlers

In [ ]:
def handle_generate_pdf(payload: dict) -> dict:
    print(f"      📄 Generating PDF for user {payload.get('user_id')}...")
    time.sleep(random.uniform(1, 3))
    return {
        'pdf_url': f"https://storage.example.com/reports/{uuid.uuid4()}.pdf",
        'pages': random.randint(5, 50)
    }

def handle_transcode_video(payload: dict) -> dict:
    print(f"      🎬 Transcoding video {payload.get('video_id')}...")
    resolutions = payload.get('resolutions', ['720p'])
    time.sleep(random.uniform(2, 4))
    return {
        'video_urls': {
            res: f"https://cdn.example.com/videos/{payload.get('video_id')}_{res}.mp4"
            for res in resolutions
        }
    }

def handle_send_bulk_email(payload: dict) -> dict:
    print(f"      📧 Sending {payload.get('recipient_count')} emails...")
    time.sleep(random.uniform(1, 2))
    return {
        'sent': payload.get('recipient_count'),
        'failed': random.randint(0, 5)
    }

print("✅ Job handlers defined!")

## 🔄 Reset & Run: Single Worker Demo

In [ ]:
reset_all()
print("✅ Ready for single worker demo")

In [ ]:
print("🚀 Running Single Worker Demo")
print("=" * 60)

def submit_job(job_type: str, payload: dict):
    job_id = str(uuid.uuid4())
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO jobs (id, job_type, payload, status)
        VALUES (%s, %s, %s, 'pending')
    """, (job_id, job_type, json.dumps(payload)))
    cursor.close()
    r.lpush(QUEUE_NAME, job_id)
    return job_id

print("\n1️⃣ Submitting 3 jobs...")
job_ids = [
    submit_job("generate_pdf", {"user_id": "user_1", "report": "annual"}),
    submit_job("transcode_video", {"video_id": "vid_1", "resolutions": ["1080p", "720p"]}),
    submit_job("send_bulk_email", {"template": "newsletter", "recipient_count": 1000})
]
print(f"   Submitted {len(job_ids)} jobs")
print(f"   Queue length: {r.llen(QUEUE_NAME)}")

print("\n2️⃣ Creating worker...")
worker = Worker("worker-1", r, conn)
worker.register_handler("generate_pdf", handle_generate_pdf)
worker.register_handler("transcode_video", handle_transcode_video)
worker.register_handler("send_bulk_email", handle_send_bulk_email)

print("\n3️⃣ Processing jobs...")
while r.llen(QUEUE_NAME) > 0:
    worker.run_once(timeout=1)

print(f"\n📊 Worker Stats:")
print(f"   Jobs processed: {worker.jobs_processed}")
print(f"   Queue length: {r.llen(QUEUE_NAME)}")

In [ ]:
print("📋 Checking Job Results")
print("=" * 60)

cursor = conn.cursor()
cursor.execute("""
    SELECT id, job_type, status, result, 
           EXTRACT(EPOCH FROM (completed_at - created_at)) as duration
    FROM jobs ORDER BY created_at
""")

for row in cursor.fetchall():
    job_id, job_type, status, result, duration = row
    status_icon = '✅' if status == 'completed' else '❌'
    print(f"\n{status_icon} Job: {str(job_id)[:8]}...")
    print(f"   Type: {job_type}")
    print(f"   Status: {status}")
    print(f"   Duration: {duration:.1f}s")
    if result:
        print(f"   Result: {result}")

cursor.close()

## 🔄 Reset & Run: Multiple Workers Demo

In [ ]:
reset_all()
print("✅ Ready for multiple workers demo")

In [ ]:
print("👥 Multiple Workers Demo")
print("=" * 60)

print("\n1️⃣ Submitting 10 jobs...")
for i in range(10):
    submit_job("generate_pdf", {"user_id": f"user_{i}", "report": "monthly"})
print(f"   Queue length: {r.llen(QUEUE_NAME)}")

def run_worker(worker_id: str):
    worker_conn = psycopg2.connect(
        host="localhost", port=5432,
        database="taskqueue", user="postgres", password="postgres"
    )
    worker_conn.autocommit = True
    
    worker_redis = redis.Redis(host='localhost', port=6379, decode_responses=True)
    
    worker = Worker(worker_id, worker_redis, worker_conn)
    worker.register_handler("generate_pdf", handle_generate_pdf)
    
    while True:
        processed = worker.run_once(timeout=1)
        if not processed and worker_redis.llen(QUEUE_NAME) == 0:
            break
    
    worker_conn.close()
    return worker.jobs_processed

print("\n2️⃣ Starting 3 workers in parallel...")
start_time = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [
        executor.submit(run_worker, f"worker-{i}")
        for i in range(3)
    ]
    results = [f.result() for f in futures]

elapsed = time.time() - start_time

print(f"\n📊 Results:")
for i, count in enumerate(results):
    print(f"   Worker {i}: processed {count} jobs")
print(f"   Total time: {elapsed:.1f}s")
print(f"   Total jobs: {sum(results)}")
print("\n✅ Jobs distributed across workers automatically!")

## 🧪 Quick Quiz

1. **Why use BRPOP instead of RPOP?**

2. **Why does each worker need its own database connection?**

3. **What happens if a worker crashes while processing?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. BRPOP vs RPOP:")
print("   - BRPOP blocks until job available")
print("   - No busy-waiting, no polling delays")
print("   - RPOP would require sleep() loops")
print()
print("2. Separate DB connections:")
print("   - Each thread needs its own connection")
print("   - psycopg2 connections aren't thread-safe")
print("   - Prevents transaction conflicts")
print()
print("3. Worker crash during processing:")
print("   - Job is marked 'processing' in DB")
print("   - Job is already removed from queue!")
print("   - Need heartbeats + timeout (next notebook)")

## 📚 Summary

### Key Takeaways

1. **BRPOP for blocking** - No busy-waiting
2. **Status updates** - pending → processing → completed/failed
3. **Multiple workers** - Scale horizontally
4. **Handler pattern** - Register handlers for job types
5. **Logging** - Track job lifecycle

### Next Up

In **Notebook 4**, we'll handle failures:
- Retry logic
- Heartbeats
- Visibility timeout